In [ ]:
import pandas as pd
import pickle
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np
from scipy.stats import chi2
import textwrap

# Dependency
* DGIDB_hypergraph
* Clustering_Result_Analysis

# Loading Variables

In [ ]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"

# Comm DGIDB genes count

In [ ]:
def DGIDB_count(c,DGIDB_genes_ncbi):
    return set(c) & set(DGIDB_genes_ncbi)

In [ ]:
with open(f"{DISEASE_FOLDER}/result_communities_ncbi_selected.pkl", "rb") as f:
    communities_ncbi_selected = pickle.load(f)
with open(f"{DISEASE_FOLDER}/result_communities_ncbi.pkl", "rb") as f:
    communities_ncbi = pickle.load(f)
with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
    DGIDB_gene_to_index = json.load(file)
DGIDB_genes = set(DGIDB_gene_to_index.keys())

In [ ]:
comm_dgidb_gene_count = {}
for i in range(len(communities_ncbi_selected)):
    dgidb_genes_set = DGIDB_count(communities_ncbi[i],DGIDB_genes)
    print(i,len(dgidb_genes_set), dgidb_genes_set)
    print(len(communities_ncbi[i]))
    comm_dgidb_gene_count[i] = len(dgidb_genes_set)
    
with open(f"../output/{DISEASE}/comm_dgidb_gene_count.json","w") as f:
    json.dump(comm_dgidb_gene_count,f)    

# Important terms processing

In [ ]:
with open(f"{DISEASE_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)

In [ ]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [ ]:
important_terms = important_terms.sort_values(
    by=["Community Index", "Adjusted P-value"],
    ascending=[True, True]
).reset_index(drop=True)

In [ ]:
important_terms

# Community Category df

In [ ]:
df = important_terms

# 2. Explode: one category per row
df_exp = df.explode("Category").reset_index(drop=True)

# 3. Parse Overlap "a/b" into numerator and denominator
_overlap = (
    df_exp["Overlap"]
    .str.split("/", expand=True)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)
df_exp["Overlap_num"] = _overlap[0]
df_exp["Overlap_den"] = _overlap[1]

# 4. Total number of terms per community
total_terms = df_exp.groupby("Community Index")["Term"].nunique()

# 5. Aggregate per (Community Index, Category)
def fisher_method(pvals):
    p = pd.Series(pvals).dropna().astype(float).clip(lower=1e-300)
    if len(p) == 0:
        return np.nan
    return float(chi2.sf(-2.0 * np.sum(np.log(p)), 2 * len(p)))

agg = (
    df_exp.groupby(["Community Index", "Category"])
    .agg(
        count=("Term", "nunique"),
        sum_a=("Overlap_num", "sum"),
        sum_b=("Overlap_den", "sum"),
        fisher_p=("Adjusted P-value", fisher_method),
    )
    .reset_index()
)

agg["Gross Overlap"] = agg["sum_a"] / agg["sum_b"]
agg["total_terms_in_comm"] = agg["Community Index"].map(total_terms)
agg["frac_of_comm"] = agg["count"] / agg["total_terms_in_comm"]

community_categories = agg[[
    "Community Index", "Category", "count", "Gross Overlap",
    "fisher_p", "total_terms_in_comm", "frac_of_comm",
]].sort_values(
    by=["Community Index", "count"],
    ascending=[True, False]
).reset_index(drop=True)

In [ ]:
community_categories["fisher_score"] = -np.log10(
    community_categories["fisher_p"].clip(lower=1e-300)
)

mn = community_categories["fisher_score"].min()
mx = community_categories["fisher_score"].max()

community_categories["fisher_norm"] = (community_categories["fisher_score"] - mn) / (mx - mn)

def safe_z(x):
    mu = x.mean()
    sd = x.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(0.0, index=x.index)   # or return x*0
    return (x - mu) / sd

community_categories["fisher_z"] = (
    community_categories
    .groupby("Community Index")["fisher_score"]
    .transform(safe_z)
)




In [ ]:
community_categories

In [ ]:
important_terms

In [ ]:
important_terms["fisher_z"] = important_terms.set_index(["Category", "Community Index"]).index.map(
    community_categories.set_index(["Category", "Community Index"])["fisher_z"]
)
important_terms.to_csv(DISEASE_FOLDER + "important_terms.csv", index=False)

In [ ]:
important_terms

In [ ]:
top20_categories = (
    community_categories.groupby("Category")["fisher_z"]
      .mean()
      .sort_values(ascending=False)
      .head(20)
      .index
)

community_categories_filtered = community_categories[community_categories["Category"].isin(top20_categories)].copy()

# Community Count Dict

In [ ]:
category_count_by_comm = {
    comm: {
        row["Category"]: (row["count"], row["Gross Overlap"])
        for _, row in group.iterrows()
    }
    for comm, group in community_categories.groupby("Community Index")
}

In [ ]:
for i in range(len(communities_selected)):
    if i not in category_count_by_comm.keys():
        category_count_by_comm[i] = {}

In [ ]:
with open(f"{DISEASE_FOLDER}/category_count_by_comm.pkl", "wb") as f:
    pickle.dump(category_count_by_comm, f)

# Heatmap with filtered categories

In [ ]:
# list of index of top 5 highest DGIDB gene count
if DISEASE != "NONE":
    sorted_comm = sorted(comm_dgidb_gene_count.items(), key=lambda x: x[1], reverse=True)
    top_5_comm_indices = [idx for idx, count in sorted_comm[:5]]
else:
    top_5_comm_indices = important_terms["Community Index"].unique().tolist()


In [ ]:
top_5_comm_indices

In [ ]:
font_size = 13
tick_font_size = 13
plt.rcParams.update({
    "font.size": font_size,
    "axes.titlesize": font_size,
    "axes.labelsize": 25,
    "xtick.labelsize": tick_font_size,
    "ytick.labelsize": tick_font_size,
    "legend.fontsize": font_size,
    "figure.titlesize": font_size,
    "legend.loc": 'best'
})


In [ ]:
heat_df = community_categories_filtered.pivot(
    index="Category",
    columns="Community Index",
    values="fisher_z"
)

label_df = community_categories_filtered.pivot(
    index="Category",
    columns="Community Index",
    values="count"
)
plt.figure(figsize=(15, 8))
ax = sns.heatmap(
    heat_df,
    cmap="viridis",
    annot=label_df,   # <-- use count for labels
    fmt=".0f",        # integer labels
    cbar=True
)

# ---- WRAP Y-AXIS LABELS ----
MAX_CHARS = 50  # tweak this
wrapped = ["\n".join(textwrap.wrap(str(s), width=MAX_CHARS)) for s in heat_df.index]
ax.set_yticklabels(wrapped, rotation=0)  # keep horizontal

plt.tight_layout()
plt.savefig(f"../../Graphs/{DISEASE}/category_gross_overlap_heatmap.pdf", bbox_inches="tight", dpi = 300)
plt.show()


# Dominant categories and terms analysis

In [ ]:
top_categories = (
    community_categories.groupby("Category")["fisher_z"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

In [ ]:
top_categories